# NB02 — Tokenisation, Dataset, DataLoader et split 90/5/5
**Projet 1.1 — Phase B, tâches 1.7 à 1.9**

| | Chemin |
|---|---|
| Entrée | `data/processed/<DATA_VERSION>/vhh_clean.csv` et `manifest.json` (produits par NB01) |
| Sorties | `train.csv`, `val.csv`, `test.csv`, `split_manifest.json` dans le même dossier |

NB02 **ne nettoie rien** : il vérifie que l'entrée est propre (sinon il s'arrête), puis tokenise et sépare. `DATA_VERSION` est choisie automatiquement (`toy` en local, `v1` sur Colab).

## Vocabulaire (V = 23)
20 acides aminés (ids 0–19, ordre `ACDEFGHIKLMNPQRSTVWY`) · `<PAD>` = 20 · `<BOS>` = 21 · `<EOS>` = 22.
Encodage : `[BOS] x₁ … x_L [EOS]`, padding à droite. Il n'y a pas de `<UNK>` : une séquence invalide lève une erreur.


In [ ]:
# ==============================================================================
# Cellule 1 : Bootstrap (local ET Google Colab), chemins, version des données
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "main"          # branche clonée sur Colab (doit contenir ce notebook et scripts/)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _find_root(start):
    """Remonte depuis `start` jusqu'au dossier qui contient utils.py."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


ROOT_DIR = _find_root(Path.cwd())

if ROOT_DIR is None and IN_COLAB:         # Colab
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)], check=True)
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError("utils.py introuvable : ouvrez le notebook dans le dépôt Recherche_AI.")

# Colab uniquement : monter le Drive AVANT get_paths() ; AI_PROJECT_DIR active le mode « données réelles »
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _drive = Path("/content/drive/MyDrive/AI_Project_Shared")
    if not _drive.exists():
        raise FileNotFoundError(f"Dossier Drive introuvable : {_drive}. Ajoutez le raccourci dans Mon Drive.")
    os.environ.setdefault("AI_PROJECT_DIR", str(_drive))

sys.path.insert(0, str(ROOT_DIR))

import json, hashlib, random
import numpy as np
import pandas as pd
import torch

from utils import (get_paths, get_data_version, set_seed, AminoAcidTokenizer, make_loader, shift,
                   is_valid_sequence, STANDARD_AA, PAD_ID, BOS_ID, EOS_ID, VOCAB_SIZE)

SEED = 42
set_seed(SEED)
BATCH_SIZE = 64
SPLIT_RATIOS = (0.90, 0.05, 0.05)

PATHS = get_paths()
DATA_VERSION = get_data_version()
DATA_DIR = PATHS["processed"] / DATA_VERSION

print(f"MODE         : {'DONNÉES RÉELLES' if DATA_VERSION != 'toy' else 'TOY (fausses données)'}")
print(f"DATA_VERSION : {DATA_VERSION}")
print(f"Dossier      : {DATA_DIR}")
print(f"Seed: {SEED} | batch size: {BATCH_SIZE} | ratios: {SPLIT_RATIOS}")


In [ ]:
# ==============================================================================
# Cellule 2 : Chargement et contrôle de l'entrée (doit être propre, sinon on s'arrête)
# ==============================================================================
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


clean_path, manifest_path = DATA_DIR / "vhh_clean.csv", DATA_DIR / "manifest.json"
for p in (clean_path, manifest_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} introuvable : exécutez d'abord NB01 (même DATA_VERSION = {DATA_VERSION}).")

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)
min_len, max_len = manifest["parameters"]["min_len"], manifest["parameters"]["max_len"]

df = pd.read_csv(clean_path, dtype=str, keep_default_na=False)
assert list(df.columns) == ["sequence"], f"colonnes inattendues : {list(df.columns)}"
sequences = df["sequence"].tolist()

invalid = [s for s in sequences if not is_valid_sequence(s, min_len, max_len)]
assert not invalid, f"{len(invalid):,} séquences invalides (alphabet ou longueur hors [{min_len}, {max_len}])"
assert len(set(sequences)) == len(sequences), "doublons exacts présents : NB01 n'a pas été exécuté correctement"

lengths = np.array([len(s) for s in sequences])
print(f"{len(sequences):,} séquences propres | longueurs [{lengths.min()}, {lengths.max()}] (bornes NB01 : [{min_len}, {max_len}])")
print(f"Produit par NB01 le {manifest['created_utc']} (commit {manifest['git_commit']}, version {manifest['data_version']})")
assert manifest["data_version"] == DATA_VERSION, "le manifest ne correspond pas à DATA_VERSION"


In [ ]:
# ==============================================================================
# Cellule 3 : Tokeniseur exact + test d'exactitude obligatoire (Tâche 1.7)
# ==============================================================================
tokenizer = AminoAcidTokenizer()
assert tokenizer.vocab_size == VOCAB_SIZE == 23
assert (tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id) == (20, 21, 22)
assert [tokenizer.aa_to_id[a] for a in STANDARD_AA] == list(range(20))
print("Vocabulaire :", {**tokenizer.aa_to_id, "<PAD>": PAD_ID, "<BOS>": BOS_ID, "<EOS>": EOS_ID})

demo = sequences[0][:30]
ids = tokenizer.encode(demo)
print(f"\nExemple : {demo}\n  ids    : {ids}\n  décodé : {tokenizer.decode(ids)}")
assert ids[0] == BOS_ID and ids[-1] == EOS_ID and tokenizer.decode(ids) == demo

# Round-trip sur 1000 séquences ALÉATOIRES (exigé par le plan) et sur 1000 séquences réelles
rng = random.Random(SEED)
random_seqs = ["".join(rng.choices(STANDARD_AA, k=rng.randint(min_len, max_len))) for _ in range(1000)]
for name, group in (("aléatoires", random_seqs), ("du jeu de données", sequences[:1000])):
    for s in group:
        assert tokenizer.decode(tokenizer.encode(s)) == s
    print(f"Round-trip OK sur {len(group)} séquences {name}")

# Une entrée invalide doit lever une erreur (pas de <UNK>)
for bad in ("ACDX", "acd", "AC-D", ""):
    try:
        tokenizer.encode(bad)
    except ValueError:
        continue
    raise AssertionError(f"séquence invalide acceptée : {bad!r}")
print("Entrées invalides correctement rejetées : 'ACDX', 'acd', 'AC-D', ''")


In [ ]:
# ==============================================================================
# Cellule 4 : Dataset, collate, DataLoader (Tâche 1.8) — vérifié sur un sous-ensemble
# ==============================================================================
demo_seqs = sequences[:2000]
loader = make_loader(demo_seqs, tokenizer, batch_size=BATCH_SIZE, shuffle=False, seed=SEED)
batch = next(iter(loader))
batch_seqs = demo_seqs[:BATCH_SIZE]
expected_len = max(len(s) for s in batch_seqs) + 2                       # BOS + résidus + EOS

assert batch.dtype == torch.long, batch.dtype
assert tuple(batch.shape) == (len(batch_seqs), expected_len), batch.shape
print(f"Batch : forme {tuple(batch.shape)} = (B, L), dtype {batch.dtype}")

for row, s in zip(batch.tolist(), batch_seqs):                           # padding uniquement à droite
    n_real = len(s) + 2
    assert row[0] == BOS_ID and row[n_real - 1] == EOS_ID
    assert all(t == PAD_ID for t in row[n_real:]), "PAD au milieu d'une séquence"
    assert PAD_ID not in row[:n_real]
print("Padding à droite uniquement, BOS en tête, EOS juste avant le padding : OK")

inputs, targets = shift(batch)
assert tuple(inputs.shape) == tuple(targets.shape) == (len(batch_seqs), expected_len - 1)
assert bool((inputs[:, 1:] == targets[:, :-1]).all()), "targets n'est pas inputs décalé de 1"
print(f"shift : inputs {tuple(inputs.shape)}, targets {tuple(targets.shape)} (targets = inputs décalé d'un cran)")


In [ ]:
# ==============================================================================
# Cellule 5 : Split aléatoire 90 / 5 / 5 à graine fixe (Tâche 1.9)
# ==============================================================================
n = len(sequences)
n_train, n_val = int(SPLIT_RATIOS[0] * n), int(SPLIT_RATIOS[1] * n)
if n_val == 0 or n - n_train - n_val == 0:
    raise ValueError(f"Seulement {n} séquences : trop peu pour un split 90/5/5")

perm = np.random.default_rng(SEED).permutation(n)
parts = {"train": perm[:n_train], "val": perm[n_train:n_train + n_val], "test": perm[n_train + n_val:]}
split_seqs = {k: [sequences[i] for i in idx] for k, idx in parts.items()}

sets = {k: set(v) for k, v in split_seqs.items()}
assert sets["train"].isdisjoint(sets["val"]) and sets["train"].isdisjoint(sets["test"]) and sets["val"].isdisjoint(sets["test"])
assert sum(len(v) for v in split_seqs.values()) == n
for k, v in split_seqs.items():
    print(f"{k:<5}: {len(v):>9,} ({len(v) / n:.1%})")
print("Les trois jeux sont disjoints et couvrent toutes les séquences.")

split_manifest = {"data_version": DATA_VERSION, "seed": SEED, "ratios": SPLIT_RATIOS,
                  "method": "random permutation after exact deduplication (NO clustering)",
                  "source": {"vhh_clean.csv": sha256_of(clean_path)}, "files": {}}
for k, v in split_seqs.items():
    path = DATA_DIR / f"{k}.csv"
    pd.DataFrame({"sequence": v}).to_csv(path, index=False)
    split_manifest["files"][f"{k}.csv"] = {"rows": len(v), "sha256": sha256_of(path)}
with open(DATA_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)
print(f"Écrit dans {DATA_DIR} : train.csv, val.csv, test.csv, split_manifest.json")


### Limite du split aléatoire (à rédiger avec vos mots pour le rapport — Tâche 1.9)
Le split est fait **après** la déduplication exacte, donc aucune séquence identique n'est partagée. Mais les **quasi-doublons** (mutations somatiques, clones apparentés) restent répartis au hasard entre train, val et test : le modèle peut avoir vu presque chaque séquence de test. Les nats/token mesurés sur val/test sont alors **optimistes**. L'amélioration prévue est un split par clusters (CD-HIT ou MMseqs2 à 90 % d'identité), où un cluster entier va dans un seul jeu. Cette limite doit apparaître dans la section « Limites » du rapport.

In [ ]:
# ==============================================================================
# Cellule 6 : Bilan de validation
# ==============================================================================
for k in ("train", "val", "test"):
    back = pd.read_csv(DATA_DIR / f"{k}.csv", dtype=str, keep_default_na=False)["sequence"].tolist()
    assert back == split_seqs[k], f"{k}.csv relu différent"
print("OK  vocabulaire V = 23, ids spéciaux 20/21/22")
print("OK  round-trip exact (1000 aléatoires + séquences réelles)")
print("OK  batch (B, L) long, padding à droite, shift cohérent")
print(f"OK  split {len(split_seqs['train']):,} / {len(split_seqs['val']):,} / {len(split_seqs['test']):,}, disjoint, relu à l'identique")
